# Section B — Genetic Algorithms

# Q4 — Implementing a Full Genetic Algorithm from Scratch

In [2]:
import random


In [3]:
# Objective function: f(x) = -x^2 + 14x + 5
# x is in {0, 1, ..., 15}, represented as a 4-bit binary list
# The global maximum is at x=7, f(7) = -(49) + 98 + 5 = 54

print("Q4: Maximise f(x) = -x^2 + 14x + 5,  x in {0..15}")
print("Chromosome: 4-bit binary list (MSB first)")
print("Global maximum: x=7, f(7) = 54")


Q4: Maximise f(x) = -x^2 + 14x + 5,  x in {0..15}
Chromosome: 4-bit binary list (MSB first)
Global maximum: x=7, f(7) = 54


In [4]:
# decode: converts a 4-bit binary list to an integer x
# chromosome[0] is the most significant bit (MSB)
# example: [1, 0, 1, 1] -> 1*8 + 0*4 + 1*2 + 1*1 = 11
def decode(chromosome):
    x = 0
    # go through each bit from MSB to LSB
    for bit in chromosome:
        x = x * 2 + bit  # shift left and add current bit
    return x


In [5]:
# fitness: returns f(x) = -x^2 + 14x + 5 for a chromosome
def fitness(chromosome):
    x = decode(chromosome)
    result = -(x * x) + 14 * x + 5
    return result


In [6]:
# roulette_select: selects one individual using fitness-proportionate selection
# the idea is like spinning a roulette wheel where each individual's slice
# is proportional to its fitness score
def roulette_select(population):
    # first, compute raw fitness for each individual
    raw_scores = []
    for ind in population:
        raw_scores.append(fitness(ind))

    # find the minimum score (we need all scores to be positive)
    min_score = raw_scores[0]
    for s in raw_scores:
        if s < min_score:
            min_score = s

    # shift scores to make them all positive
    if min_score <= 0:
        offset = abs(min_score) + 1
    else:
        offset = 0

    adjusted = []
    for s in raw_scores:
        adjusted.append(s + offset)

    # calculate total fitness
    total = 0
    for s in adjusted:
        total = total + s

    # spin the roulette wheel
    spin = random.random()  # random number between 0.0 and 1.0
    cumulative = 0.0

    for i in range(len(population)):
        cumulative = cumulative + (adjusted[i] / total)
        if spin <= cumulative:
            return population[i]  # this individual is selected

    # fallback (just in case floating point causes issues)
    return population[-1]


In [7]:
# single_point_crossover: combines two parent chromosomes at a given point
# offspring1 = parent1[:point] + parent2[point:]
# offspring2 = parent2[:point] + parent1[point:]
def single_point_crossover(parent1, parent2, point):
    # take first 'point' bits from parent1, rest from parent2
    offspring1 = []
    for i in range(len(parent1)):
        if i < point:
            offspring1.append(parent1[i])
        else:
            offspring1.append(parent2[i])

    # take first 'point' bits from parent2, rest from parent1
    offspring2 = []
    for i in range(len(parent2)):
        if i < point:
            offspring2.append(parent2[i])
        else:
            offspring2.append(parent1[i])

    return offspring1, offspring2


In [8]:
# mutate: flip each bit independently with probability p_m
def mutate(chromosome, p_m):
    new_chromosome = []
    for bit in chromosome:
        if random.random() < p_m:
            # flip this bit
            new_chromosome.append(1 - bit)
        else:
            # keep this bit the same
            new_chromosome.append(bit)
    return new_chromosome


In [9]:
# init_population: create a random population
def init_population(pop_size, bits=4):
    population = []
    for i in range(pop_size):
        # create one random chromosome
        chromosome = []
        for j in range(bits):
            chromosome.append(random.randint(0, 1))
        population.append(chromosome)
    return population


In [10]:
# PART (a): Test each function in isolation
print("PART (a) - Core GA Components: decode, fitness, roulette_select,")
print("           single_point_crossover, mutate")
print()
print("""
Objective function  : f(x) = -x^2 + 14x + 5,   x in {0, 1, ..., 15}
Chromosome          : 4-bit binary list, MSB first
Global maximum      : x = 7,  f(7) = -(49) + 98 + 5 = 54
Test chromosomes    : [0,1,1,0], [1,0,0,1], [1,1,0,0], [0,0,1,1]
""")

test_chroms = [
    [0, 1, 1, 0],   # x = 6,  f = 53
    [1, 0, 0, 1],   # x = 9,  f = 50
    [1, 1, 0, 0],   # x = 12, f = 29
    [0, 0, 1, 1],   # x = 3,  f = 38
]

# TEST 1: decode()
print("--- TEST 1: decode(chromosome) ---")
print(f"  Rule: MSB first -> value = b3*8 + b2*4 + b1*2 + b0*1")
print()
print(f"  {'Chromosome':<20}  {'Working':<40}  {'x':>4}")
print(f"  {'-'*70}")

expected_x = [6, 9, 12, 3]
for i in range(len(test_chroms)):
    chrom = test_chroms[i]
    exp = expected_x[i]
    x = decode(chrom)
    working = f"{chrom[0]}*8 + {chrom[1]}*4 + {chrom[2]}*2 + {chrom[3]}*1 = {x}"
    if x == exp:
        ok = "OK"
    else:
        ok = f"ERROR (expected {exp})"
    print(f"  {str(chrom):<20}  {working:<40}  {x:>4}  {ok}")

# TEST 2: fitness()
print()
print("--- TEST 2: fitness(chromosome)  [f(x) = -x^2 + 14x + 5] ---")
print()
print(f"  {'Chromosome':<20}  {'x':>4}  {'Working':<40}  {'f(x)':>6}")

for chrom in test_chroms:
    x = decode(chrom)
    fv = fitness(chrom)
    working = f"-({x})^2 + 14*{x} + 5 = -{x*x} + {14*x} + 5 = {fv}"
    print(f"  {str(chrom):<20}  {x:>4}  {working:<40}  {fv:>6}")

print(f"\n  Reference: x=7 -> f(7) = -(49) + 98 + 5 = 54  [GLOBAL MAXIMUM]")

# TEST 3: roulette_select()
print()
print("--- TEST 3: roulette_select(population) ---")
print()

f_vals = []
for chrom in test_chroms:
    f_vals.append(fitness(chrom))

total_f = 0
for fv in f_vals:
    total_f = total_f + fv

print(f"  Population fitness values : {f_vals}")
print(f"  Total fitness sum         : {total_f}")
print()
print("  Selection probabilities (P = f / total_f):")
print(f"  {'Chromosome':<20}  {'f(x)':>6}  {'f/total':>10}  {'Cumulative':>12}")

cum = 0.0
for i in range(len(test_chroms)):
    p = f_vals[i] / total_f
    cum = cum + p
    print(f"  {str(test_chroms[i]):<20}  {f_vals[i]:>6}  {p:>10.4f}  {cum:>12.4f}")

print()
print("  Empirical check -- 10,000 spins (seed=42):")
random.seed(42)
counts = [0, 0, 0, 0]
for spin_i in range(10000):
    sel = roulette_select(test_chroms)
    for i in range(len(test_chroms)):
        if sel == test_chroms[i]:
            counts[i] = counts[i] + 1
            break

print(f"  {'Chromosome':<20}  {'Theory':>8}  {'Empirical':>10}  {'Match?':>8}")
print(f"  {'-'*52}")
for i in range(len(test_chroms)):
    theory = f_vals[i] / total_f
    emp = counts[i] / 10000
    if abs(theory - emp) < 0.02:
        match = "YES"
    else:
        match = "DEVIATION"
    print(f"  {str(test_chroms[i]):<20}  {theory:>8.4f}  {emp:>10.4f}  {match:>8}")

# TEST 4: single_point_crossover()
print()
print("--- TEST 4: single_point_crossover(parent1, parent2, point) ---")
print()
p1 = [0, 1, 1, 0]
p2 = [1, 0, 0, 1]
print(f"  Parent1 = {p1}   (x={decode(p1)}, f={fitness(p1)})")
print(f"  Parent2 = {p2}   (x={decode(p2)}, f={fitness(p2)})")
print()
print(f"  {'Point':<8}  {'Offspring1':<20}  {'x':>4}  {'f':>4}  {'Offspring2':<20}  {'x':>4}  {'f':>4}")

for pt in [1, 2, 3]:
    o1, o2 = single_point_crossover(p1, p2, pt)
    if decode(o1) == 7:
        note1 = " <- global max!"
    else:
        note1 = ""
    if decode(o2) == 7:
        note2 = " <- global max!"
    else:
        note2 = ""
    print(f"  {pt:<8}  {str(o1):<20}  {decode(o1):>4}  {fitness(o1):>4}{note1}"
          f"  {str(o2):<20}  {decode(o2):>4}  {fitness(o2):>4}{note2}")

# TEST 5: mutate()
print()
print("--- TEST 5: mutate(chromosome, p_m) ---")
print()
base = [0, 1, 1, 0]
print(f"  Base chromosome: {base}  (x={decode(base)}, f={fitness(base)})")
print()
print(f"  {'p_m':<6}  {'Expected flips/run':<22}  {'Sample results (5 runs, seed=10)'}")

for p_m in [0.0, 0.1, 0.5, 1.0]:
    random.seed(10)
    samples = []
    flip_cnts = []
    for r in range(5):
        s = mutate(base, p_m)
        samples.append(s)
        # count how many bits flipped
        flips = 0
        for b_idx in range(len(base)):
            if base[b_idx] != s[b_idx]:
                flips = flips + 1
        flip_cnts.append(flips)

    exp_flips = f"~{4 * p_m:.1f} bits"
    sample_str = "  ".join(f"{samples[k]}({flip_cnts[k]}f)" for k in range(5))
    print(f"  {p_m:<6}  {exp_flips:<22}  {sample_str}")

# SUMMARY TABLE
print()
print("--- SUMMARY: All Test Chromosomes ---")
print()
total_f2 = 0
for chrom in test_chroms:
    total_f2 = total_f2 + fitness(chrom)

print(f"  {'Individual':<12}  {'Chromosome':<20}  {'x':>4}  {'f(x)':>6}  {'Sel. Prob':>10}")
for i in range(len(test_chroms)):
    chrom = test_chroms[i]
    x = decode(chrom)
    fv = fitness(chrom)
    sp = fv / total_f2
    print(f"  P{i+1:<11}  {str(chrom):<20}  {x:>4}  {fv:>6}  {sp:>10.4f}")

print(f"\n  Total fitness = {total_f2}")
print(f"  Target (global max): chromosome [0,1,1,1] -> x=7, f=54")
print(f"  Note: none of the four test chromosomes encode x=7.")


PART (a) - Core GA Components: decode, fitness, roulette_select,
           single_point_crossover, mutate


Objective function  : f(x) = -x^2 + 14x + 5,   x in {0, 1, ..., 15}
Chromosome          : 4-bit binary list, MSB first
Global maximum      : x = 7,  f(7) = -(49) + 98 + 5 = 54
Test chromosomes    : [0,1,1,0], [1,0,0,1], [1,1,0,0], [0,0,1,1]

--- TEST 1: decode(chromosome) ---
  Rule: MSB first -> value = b3*8 + b2*4 + b1*2 + b0*1

  Chromosome            Working                                      x
  ----------------------------------------------------------------------
  [0, 1, 1, 0]          0*8 + 1*4 + 1*2 + 0*1 = 6                    6  OK
  [1, 0, 0, 1]          1*8 + 0*4 + 0*2 + 1*1 = 9                    9  OK
  [1, 1, 0, 0]          1*8 + 1*4 + 0*2 + 0*1 = 12                  12  OK
  [0, 0, 1, 1]          0*8 + 0*4 + 1*2 + 1*1 = 3                    3  OK

--- TEST 2: fitness(chromosome)  [f(x) = -x^2 + 14x + 5] ---

  Chromosome               x  Working              

In [11]:
# PART (b): Full run_ga() function
def run_ga(pop_size, num_generations, p_m, elitism, verbose=True):
    """
    Run a complete Genetic Algorithm.
    pop_size        : number of individuals in population
    num_generations : how many generations to evolve
    p_m             : per-bit mutation probability
    elitism         : if True, best individual always survives to next gen
    verbose         : if True, print generation-by-generation table
    Returns history: list of (generation, best_fitness, best_x)
    """

    # start with a random population
    population = init_population(pop_size)
    history = []

    for gen in range(1, num_generations + 1):

        # evaluate each individual
        scored = []
        for ind in population:
            x = decode(ind)
            fv = fitness(ind)
            scored.append((ind, x, fv))

        # sort by fitness (highest first)
        # simple bubble sort to avoid using .sort() with lambda
        for i in range(len(scored)):
            for j in range(i + 1, len(scored)):
                if scored[j][2] > scored[i][2]:
                    scored[i], scored[j] = scored[j], scored[i]

        best_ind = scored[0][0]
        best_x = scored[0][1]
        best_fit = scored[0][2]
        history.append((gen, best_fit, best_x))

        if verbose:
            print(f"\n  Generation {gen}")
            print(f"  {'Rank':<5}  {'Chromosome':<20}  {'x':>4}  {'f(x)':>6}  {'Note'}")
            for rank in range(len(scored)):
                ind = scored[rank][0]
                x = scored[rank][1]
                fv = scored[rank][2]
                note = ""
                if rank == 0:
                    note = "BEST"
                if x == 7:
                    if note != "":
                        note = note + " | GLOBAL MAX"
                    else:
                        note = "GLOBAL MAX"
                print(f"  {rank+1:<5}  {str(ind):<20}  {x:>4}  {fv:>6}  {note}")
            print(f"  Best this generation: x={best_x}, f={best_fit}")

        # build next generation
        next_pop = []

        # if elitism is on, carry the best individual unchanged
        if elitism:
            next_pop.append(list(best_ind))

        # fill the rest with offspring from crossover + mutation
        while len(next_pop) < pop_size:
            # select two parents using roulette
            p1 = roulette_select(population)
            p2 = roulette_select(population)

            # pick a random crossover point
            point = random.randint(1, 3)

            # create two offspring
            o1, o2 = single_point_crossover(p1, p2, point)

            # mutate both offspring
            o1 = mutate(o1, p_m)
            o2 = mutate(o2, p_m)

            next_pop.append(o1)
            if len(next_pop) < pop_size:
                next_pop.append(o2)

        # replace old population with new one
        population = next_pop

    return history


In [12]:
# PART (b): Run the GA with specific settings and print generation-by-generation trace
print("PART (b) - run_ga(): Generation-by-Generation Trace")
print("           pop_size=4, num_generations=10, p_m=0.1, elitism=False")
print()

print("""
Algorithm overview:
  1. Initialise a random population of pop_size=4 chromosomes (4 bits each).
  2. Each generation:
       a) Evaluate fitness of every individual.
       b) Select two parents via roulette wheel selection.
       c) Apply single-point crossover at a random bit position.
       d) Apply per-bit mutation at rate p_m=0.1.
       e) Replace the old population with the new offspring.
  3. Track the best individual per generation.
  Elitism=False means the best individual is NOT guaranteed to survive.
""")

random.seed(42)
history = run_ga(pop_size=4, num_generations=10, p_m=0.1, elitism=False, verbose=True)

print()
print("  Convergence Summary")
print()
print(f"  {'Gen':>4}  {'Best x':>7}  {'Best f(x)':>10}  {'Progress'}")
prev_best = None

for i in range(len(history)):
    gen = history[i][0]
    bf = history[i][1]
    bx = history[i][2]

    arrow = ""
    if prev_best is None or bf > prev_best:
        arrow = " ^ improved"

    if bx == 7:
        gm = " [GLOBAL MAX]"
    else:
        gm = ""

    print(f"  {gen:>4}  {bx:>7}  {bf:>10}{arrow}{gm}")
    prev_best = bf

# check if global optimum was ever found
max_fit_found = history[0][1]
found_optimum = False
for h in history:
    if h[1] > max_fit_found:
        max_fit_found = h[1]
    if h[2] == 7:
        found_optimum = True

if found_optimum:
    print(f"\n  Global optimum (x=7, f=54) found : YES")
else:
    print(f"\n  Global optimum (x=7, f=54) found : NO")
print(f"  Maximum fitness reached          : {max_fit_found}")


PART (b) - run_ga(): Generation-by-Generation Trace
           pop_size=4, num_generations=10, p_m=0.1, elitism=False


Algorithm overview:
  1. Initialise a random population of pop_size=4 chromosomes (4 bits each).
  2. Each generation:
       a) Evaluate fitness of every individual.
       b) Select two parents via roulette wheel selection.
       c) Apply single-point crossover at a random bit position.
       d) Apply per-bit mutation at rate p_m=0.1.
       e) Replace the old population with the new offspring.
  3. Track the best individual per generation.
  Elitism=False means the best individual is NOT guaranteed to survive.


  Generation 1
  Rank   Chromosome               x    f(x)  Note
  1      [1, 0, 0, 0]             8      53  BEST
  2      [0, 0, 1, 0]             2      29  
  3      [0, 0, 0, 0]             0       5  
  4      [0, 0, 0, 0]             0       5  
  Best this generation: x=8, f=53

  Generation 2
  Rank   Chromosome               x    f(x)  Note
  1 

In [13]:
print("""
Analysis:

  Population structure (pop_size=4):
    With only 4 individuals, diversity is extremely limited. Each chromosome
    covers one of 16 possible x values, so 4 individuals represent just 25%
    of the search space per generation. Despite this, roulette selection
    naturally steers offspring toward higher-fitness individuals.

  Role of crossover:
    Single-point crossover at a random position [1,2,3] recombines bit patterns
    from two parents. In a 4-bit space this is powerful -- crossover at point=3
    between [0,1,1,0] and [1,0,0,1] produces [0,1,1,1] which encodes x=7,
    the global maximum. The GA assembles good bit-blocks (schemata) from
    different parents, gradually improving the population.

  Role of mutation at p_m=0.1:
    Expected flips per chromosome = 4 * 0.1 = 0.4, meaning most chromosomes
    are unchanged by mutation each generation. This is deliberate: low mutation
    maintains the diversity generated by crossover without disrupting solutions
    already found. Mutation's main role is to re-introduce diversity if the
    population converges prematurely.

  Without elitism, regression can occur:
    A good individual found in generation k can be lost in generation k+1 if
    roulette selection does not pick it as a parent. This is visible in the
    convergence summary above as "non-monotone" fitness curves. Elitism, tested
    in Part (c), guarantees the best solution is always preserved.
""")



Analysis:

  Population structure (pop_size=4):
    With only 4 individuals, diversity is extremely limited. Each chromosome
    covers one of 16 possible x values, so 4 individuals represent just 25%
    of the search space per generation. Despite this, roulette selection
    naturally steers offspring toward higher-fitness individuals.

  Role of crossover:
    Single-point crossover at a random position [1,2,3] recombines bit patterns
    from two parents. In a 4-bit space this is powerful -- crossover at point=3
    between [0,1,1,0] and [1,0,0,1] produces [0,1,1,1] which encodes x=7,
    the global maximum. The GA assembles good bit-blocks (schemata) from
    different parents, gradually improving the population.

  Role of mutation at p_m=0.1:
    Expected flips per chromosome = 4 * 0.1 = 0.4, meaning most chromosomes
    are unchanged by mutation each generation. This is deliberate: low mutation
    maintains the diversity generated by crossover without disrupting solutions
   

In [14]:
# PART (c) - EXPERIMENT 1: Elitism=False vs Elitism=True
print("PART (c) - EXPERIMENT 1: Elitism=False vs Elitism=True")
print("           pop_size=4, num_generations=20, p_m=0.1, 30 trials")
print()

print("""
What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54, the global optimum) was ever found.
  - The generation number where f >= 50 was FIRST achieved (or Never).
""")

POP_SIZE = 4
NUM_GENS = 20
P_M = 0.1
TRIALS = 30


# run one condition (elitism on or off) for 30 trials
def run_condition(elitism_flag, seed_offset):
    results = []
    for t in range(TRIALS):
        random.seed(seed_offset + t)
        history = run_ga(POP_SIZE, NUM_GENS, P_M, elitism_flag, verbose=False)

        # find best fitness across all generations
        best_f = history[0][1]
        for h in history:
            if h[1] > best_f:
                best_f = h[1]

        # did we ever find x=7?
        found = False
        for h in history:
            if h[2] == 7:
                found = True
                break

        # first generation where f >= 50
        gen_50 = None
        for h in history:
            if h[1] >= 50:
                gen_50 = h[0]
                break

        results.append((best_f, found, gen_50))
    return results


print("[c.1] Running Elitism=False (30 trials, seeds 0-29)...")
res_no = run_condition(False, 0)

print("[c.2] Running Elitism=True  (30 trials, seeds 1000-1029)...")
res_yes = run_condition(True, 1000)

# print per-trial results for each condition
for label, results in [("Elitism=False", res_no), ("Elitism=True", res_yes)]:
    print()
    print(f" {label}: Per-Trial Results")
    print(f"  {'Trial':>6}  {'Best f':>7}  {'Found x=7?':>11}  {'First gen f>=50':>16}")

    for t in range(len(results)):
        bf = results[t][0]
        found = results[t][1]
        g50 = results[t][2]

        if g50 is not None:
            g_str = f"Gen {g50}"
        else:
            g_str = "Never"

        if found:
            opt_str = "YES"
        else:
            opt_str = "No"

        print(f"  {t+1:>6}  {bf:>7}  {opt_str:>11}  {g_str:>16}")


PART (c) - EXPERIMENT 1: Elitism=False vs Elitism=True
           pop_size=4, num_generations=20, p_m=0.1, 30 trials


What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54, the global optimum) was ever found.
  - The generation number where f >= 50 was FIRST achieved (or Never).

[c.1] Running Elitism=False (30 trials, seeds 0-29)...
[c.2] Running Elitism=True  (30 trials, seeds 1000-1029)...

 Elitism=False: Per-Trial Results
   Trial   Best f   Found x=7?   First gen f>=50
       1       54          YES             Gen 1
       2       54          YES            Gen 11
       3       54          YES             Gen 1
       4       54          YES             Gen 1
       5       54          YES             Gen 1
       6       54          YES             Gen 2
       7       54          YES             Gen 1
       8       53           No             Gen 1
       9       54          YES             Gen 1
      10       54          YES 

In [15]:
# aggregate statistics for experiment 1
print()
print(" Aggregate Comparison Table")
print()

# calculate stats for no-elitism
total_f_no = 0
n_opt_no = 0
gens_hit_no = []
for result in res_no:
    bf = result[0]
    found = result[1]
    g50 = result[2]
    total_f_no = total_f_no + bf
    if found:
        n_opt_no = n_opt_no + 1
    if g50 is not None:
        gens_hit_no.append(g50)

avg_f_no = total_f_no / TRIALS
if len(gens_hit_no) > 0:
    total_g_no = 0
    for g in gens_hit_no:
        total_g_no = total_g_no + g
    avg_g_no = total_g_no / len(gens_hit_no)
    g_no_str = f"{avg_g_no:.2f} ({len(gens_hit_no)}/30 runs)"
else:
    avg_g_no = None
    g_no_str = "N/A"

# calculate stats for with-elitism
total_f_yes = 0
n_opt_yes = 0
gens_hit_yes = []
for result in res_yes:
    bf = result[0]
    found = result[1]
    g50 = result[2]
    total_f_yes = total_f_yes + bf
    if found:
        n_opt_yes = n_opt_yes + 1
    if g50 is not None:
        gens_hit_yes.append(g50)

avg_f_yes = total_f_yes / TRIALS
if len(gens_hit_yes) > 0:
    total_g_yes = 0
    for g in gens_hit_yes:
        total_g_yes = total_g_yes + g
    avg_g_yes = total_g_yes / len(gens_hit_yes)
    g_yes_str = f"{avg_g_yes:.2f} ({len(gens_hit_yes)}/30 runs)"
else:
    avg_g_yes = None
    g_yes_str = "N/A"

# find min and max best fitness
min_f_no = res_no[0][0]
max_f_no = res_no[0][0]
for result in res_no:
    if result[0] < min_f_no:
        min_f_no = result[0]
    if result[0] > max_f_no:
        max_f_no = result[0]

min_f_yes = res_yes[0][0]
max_f_yes = res_yes[0][0]
for result in res_yes:
    if result[0] < min_f_yes:
        min_f_yes = result[0]
    if result[0] > max_f_yes:
        max_f_yes = result[0]

print(f"  {'Metric':<40}  {'Elitism=False':>14}  {'Elitism=True':>13}")
print(f"  {'Average best fitness (30 trials)':<40}  {avg_f_no:>14.2f}  {avg_f_yes:>13.2f}")
print(f"  {'Runs that found x=7 (f=54)':<40}  {str(n_opt_no)+'/30':>14}  {str(n_opt_yes)+'/30':>13}")
print(f"  {'Avg gen to first f>=50':<40}  {g_no_str:>14}  {g_yes_str:>13}")
print(f"  {'Min best fitness across trials':<40}  {min_f_no:>14}  {min_f_yes:>13}")
print(f"  {'Max best fitness across trials':<40}  {max_f_no:>14}  {max_f_yes:>13}")

# Per-generation average best fitness table
print()
print(" Per-Generation Avg Best Fitness (averaged over 30 trials)")
print()
print(f"  {'Gen':>4}  {'Avg f (No Elitism)':>20}  {'Avg f (Elitism)':>17}")

# run again to get all histories
all_no = []
for i in range(TRIALS):
    random.seed(i)
    h = run_ga(POP_SIZE, NUM_GENS, P_M, False, verbose=False)
    all_no.append(h)

all_yes = []
for i in range(TRIALS):
    random.seed(1000 + i)
    h = run_ga(POP_SIZE, NUM_GENS, P_M, True, verbose=False)
    all_yes.append(h)

for g_idx in range(NUM_GENS):
    # compute average for no elitism
    total_no = 0
    for t in range(TRIALS):
        total_no = total_no + all_no[t][g_idx][1]
    avg_no = total_no / TRIALS

    # compute average for with elitism
    total_yes = 0
    for t in range(TRIALS):
        total_yes = total_yes + all_yes[t][g_idx][1]
    avg_yes = total_yes / TRIALS

    print(f"  {g_idx+1:>4}  {avg_no:>20.2f}  {avg_yes:>17.2f}")



 Aggregate Comparison Table

  Metric                                     Elitism=False   Elitism=True
  Average best fitness (30 trials)                   53.87          53.53
  Runs that found x=7 (f=54)                         26/30          16/30
  Avg gen to first f>=50                    1.53 (30/30 runs)  1.63 (30/30 runs)
  Min best fitness across trials                        53             53
  Max best fitness across trials                        54             54

 Per-Generation Avg Best Fitness (averaged over 30 trials)

   Gen    Avg f (No Elitism)    Avg f (Elitism)
     1                 49.37              47.60
     2                 50.70              49.33
     3                 50.20              51.47
     4                 49.00              51.77
     5                 50.53              51.87
     6                 49.33              52.20
     7                 50.47              52.20
     8                 51.17              52.40
     9                 51.

In [16]:
# PART (c) - EXPERIMENT 2: Mutation Rate Comparison
print("PART (c) - EXPERIMENT 2: Mutation Rate Comparison")
print("           pop_size=4, num_generations=20, elitism=False, 30 trials")
print()

print("""
What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54) was found.
  - Average generation where f >= 50 was first achieved.
Mutation rates tested: p_m in {0.01, 0.1, 0.3, 0.5}
""")

POP_SIZE = 4
NUM_GENS = 20
TRIALS = 30
PM_LIST = [0.01, 0.1, 0.3, 0.5]

# store results for each mutation rate
all_histories = {}
agg_data = {}

for pm_idx in range(len(PM_LIST)):
    p_m = PM_LIST[pm_idx]

    best_fs = []
    found_opts = []
    gen_50s = []
    histories = []

    for t in range(TRIALS):
        random.seed(pm_idx * 100 + t)
        history = run_ga(POP_SIZE, NUM_GENS, p_m, elitism=False, verbose=False)
        histories.append(history)

        # find best fitness
        best_f = history[0][1]
        for h in history:
            if h[1] > best_f:
                best_f = h[1]

        # did we find x=7?
        found = False
        for h in history:
            if h[2] == 7:
                found = True
                break

        # first gen with f >= 50
        gen_50 = None
        for h in history:
            if h[1] >= 50:
                gen_50 = h[0]
                break

        best_fs.append(best_f)
        found_opts.append(found)
        gen_50s.append(gen_50)

    all_histories[p_m] = histories

    # compute aggregates
    total_bf = 0
    for bf in best_fs:
        total_bf = total_bf + bf
    avg_f = total_bf / TRIALS

    n_opt = 0
    for f in found_opts:
        if f:
            n_opt = n_opt + 1

    gens_hit = []
    for g in gen_50s:
        if g is not None:
            gens_hit.append(g)

    if len(gens_hit) > 0:
        total_gens = 0
        for g in gens_hit:
            total_gens = total_gens + g
        avg_g50 = total_gens / len(gens_hit)
    else:
        avg_g50 = None

    min_f = best_fs[0]
    max_f = best_fs[0]
    for bf in best_fs:
        if bf < min_f:
            min_f = bf
        if bf > max_f:
            max_f = bf

    agg_data[p_m] = {
        "avg_f": avg_f,
        "n_opt": n_opt,
        "avg_g50": avg_g50,
        "n_g50": len(gens_hit),
        "min_f": min_f,
        "max_f": max_f,
        "best_fs": best_fs,
    }


PART (c) - EXPERIMENT 2: Mutation Rate Comparison
           pop_size=4, num_generations=20, elitism=False, 30 trials


What we measure per trial:
  - Best fitness reached across all 20 generations.
  - Whether x=7 (f=54) was found.
  - Average generation where f >= 50 was first achieved.
Mutation rates tested: p_m in {0.01, 0.1, 0.3, 0.5}



In [17]:
# print per-rate detail tables
for p_m in PM_LIST:
    histories_pm = all_histories[p_m]
    print(f"  p_m={p_m}: Per-Trial Results")
    print(f"  {'Trial':>6}  {'Best f':>7}  {'Found x=7?':>11}  {'First gen f>=50':>16}")
    print(f"  {'-'*47}")

    for t in range(TRIALS):
        history = histories_pm[t]

        best_f_t = history[0][1]
        for h in history:
            if h[1] > best_f_t:
                best_f_t = h[1]

        found_t = False
        for h in history:
            if h[2] == 7:
                found_t = True
                break

        gen_50_t = None
        for h in history:
            if h[1] >= 50:
                gen_50_t = h[0]
                break

        if gen_50_t is not None:
            g_str = f"Gen {gen_50_t}"
        else:
            g_str = "Never"

        if found_t:
            o_str = "YES"
        else:
            o_str = "No"

        print(f"  {t+1:>6}  {best_f_t:>7}  {o_str:>11}  {g_str:>16}")
    print()

# aggregate comparison table
print()
print("  Aggregate Comparison Table")
print()
print(f"  {'p_m':>6}  {'Avg Best f':>11}  {'Found x=7':>11}  {'Avg Gen f>=50':>15}  {'Min f':>7}  {'Max f':>7}")
print(f"  {'-'*66}")

for p_m in PM_LIST:
    d = agg_data[p_m]
    if d["avg_g50"] is not None:
        g_str = f"{d['avg_g50']:.1f} ({d['n_g50']}/30)"
    else:
        g_str = "N/A"
    print(f"  {p_m:>6}  {d['avg_f']:>11.2f}  {str(d['n_opt'])+'/30':>11}  {g_str:>15}  {d['min_f']:>7}  {d['max_f']:>7}")

# fitness distribution table
print()
print("  --- Best Fitness Distribution Across 30 Trials ---")
print()
header = f"  {'f value':>8}"
for p_m in PM_LIST:
    header = header + f"  {'p_m='+str(p_m):>10}"
print(header)
print(f"  {'-'*(10 + 14 * len(PM_LIST))}")

for fv in [54, 53, 50, 45, 38, 29]:
    row = f"  {fv:>8}"
    for p_m in PM_LIST:
        cnt = 0
        for bf in agg_data[p_m]["best_fs"]:
            if bf == fv:
                cnt = cnt + 1
        if cnt > 0:
            row = row + f"  {cnt:>10}"
        else:
            row = row + f"  {'-':>10}"
    print(row)

# per-generation avg best fitness trace
print()
print("  Avg Best Fitness Per Generation (30-trial avg)")
print()
header = f"  {'Gen':>4}"
for p_m in PM_LIST:
    header = header + f"  {'p_m='+str(p_m):>12}"
print(header)
print(f"  {'-'*(6 + 14 * len(PM_LIST))}")

for g_idx in range(NUM_GENS):
    row = f"  {g_idx+1:>4}"
    for p_m in PM_LIST:
        total = 0
        for t in range(TRIALS):
            total = total + all_histories[p_m][t][g_idx][1]
        avg = total / TRIALS
        row = row + f"  {avg:>12.2f}"
    print(row)


  p_m=0.01: Per-Trial Results
   Trial   Best f   Found x=7?   First gen f>=50
  -----------------------------------------------
       1       53           No             Gen 1
       2       38           No             Never
       3       54          YES             Gen 1
       4       53           No             Gen 1
       5       54          YES             Gen 1
       6       53           No             Gen 2
       7       54          YES             Gen 1
       8       53           No             Gen 1
       9       53           No             Gen 1
      10       54          YES             Gen 1
      11       54          YES             Gen 1
      12       53           No             Gen 1
      13       53           No             Gen 1
      14       53           No             Gen 1
      15       50           No             Gen 9
      16       53           No             Gen 1
      17       54          YES             Gen 1
      18       54          YES        

In [18]:
print("""
Analysis (4-5 sentences):
  Elitism significantly improves both convergence speed and solution quality
  because it guarantees the best solution found so far is never lost to random
  selection; the average best fitness with elitism is higher and the number of
  runs finding x=7 is greater than without it. Among mutation rates, p_m=0.1
  typically performs best because it provides just enough bit diversity to
  escape local optima without disrupting chromosomes that are already near-
  optimal. A very high p_m (0.3-0.5) is harmful because it destroys too many
  good bit combinations each generation, making the GA behave more like random
  search than directed evolution. The per-generation average best fitness
  table confirms that elitism produces a monotonically non-decreasing best
  fitness curve, while no-elitism shows occasional drops (regression).
""")



Analysis (4-5 sentences):
  Elitism significantly improves both convergence speed and solution quality
  because it guarantees the best solution found so far is never lost to random
  selection; the average best fitness with elitism is higher and the number of
  runs finding x=7 is greater than without it. Among mutation rates, p_m=0.1
  typically performs best because it provides just enough bit diversity to
  escape local optima without disrupting chromosomes that are already near-
  optimal. A very high p_m (0.3-0.5) is harmful because it destroys too many
  good bit combinations each generation, making the GA behave more like random
  search than directed evolution. The per-generation average best fitness
  table confirms that elitism produces a monotonically non-decreasing best
  fitness curve, while no-elitism shows occasional drops (regression).



In [19]:
# PART (d): Manual Trace - One Complete GA Generation (no run_ga() called)
print("PART (d) - Manual Trace: One Complete GA Generation")
print("           No code-based GA calls -- all arithmetic shown explicitly")
print()

print("""
Starting population and given random numbers:
  P1 = [0,1,1,0]    r1 = 0.12  (selection spin 1)
  P2 = [1,0,0,1]    r2 = 0.47  (selection spin 2)
  P3 = [1,1,0,0]    r3 = 0.68  (selection spin 3)
  P4 = [0,0,1,1]    r4 = 0.91  (selection spin 4)
  Crossover point = 2 (after bit index 2)
  Mutation on O1 : per-bit random values = [0.08, 0.43, 0.91, 0.05], p_m = 0.1
""")

# STEP 1: Decode chromosomes and compute f(x)
print(" STEP 1: Decode chromosomes and compute f(x) = -x^2 + 14x + 5")
print()

pop_chroms = {
    "P1": [0, 1, 1, 0],
    "P2": [1, 0, 0, 1],
    "P3": [1, 1, 0, 0],
    "P4": [0, 0, 1, 1],
}

decoded = {}
for name in ["P1", "P2", "P3", "P4"]:
    chrom = pop_chroms[name]
    # manual decode
    x = chrom[0]*8 + chrom[1]*4 + chrom[2]*2 + chrom[3]*1
    fv = -(x*x) + 14*x + 5
    decoded[name] = (chrom, x, fv)

    work_x = f"{chrom[0]}*8 + {chrom[1]}*4 + {chrom[2]}*2 + {chrom[3]}*1 = {x}"
    work_f = f"-{x}^2 + 14*{x} + 5 = -{x*x} + {14*x} + 5 = {fv}"
    print(f"  {name} = {chrom}")
    print(f"       decode  : {work_x}")
    print(f"       f(x)    : {work_f}")

total_f = 0
for name in ["P1", "P2", "P3", "P4"]:
    total_f = total_f + decoded[name][2]

vals = [decoded[n][2] for n in ["P1","P2","P3","P4"]]
print(f"\n  Total fitness Sf = {vals[0]} + {vals[1]} + {vals[2]} + {vals[3]} = {total_f}")

# STEP 2: Selection probabilities
print()
print(" STEP 2: Selection probabilities and cumulative ranges")
print()
print("  P(select) = f(x) / Sf")
print()
print(f"  {'Individual':<12}  {'Chromosome':<16}  {'x':>4}  {'f(x)':>6}  {'P(sel)':>8}  {'Cumulative':>12}")

cum = 0.0
cum_data = {}
for name in ["P1", "P2", "P3", "P4"]:
    chrom = decoded[name][0]
    x = decoded[name][1]
    fv = decoded[name][2]
    p = fv / total_f
    lo = cum
    cum = cum + p
    cum_data[name] = (chrom, x, fv, p, lo, cum)
    print(f"  {name:<12}  {str(chrom):<16}  {x:>4}  {fv:>6}  {fv}/{total_f}={p:.4f}  {lo:.4f} - {cum:.4f}")

print()
print("  Roulette ranges:")
for name in ["P1", "P2", "P3", "P4"]:
    lo = cum_data[name][4]
    hi = cum_data[name][5]
    print(f"    {name}: [{lo:.4f}, {hi:.4f})")

# STEP 3: Select parents using r1-r4
print()
print(" STEP 3: Parent selection using r1=0.12, r2=0.47, r3=0.68, r4=0.91")
print()

spins = [0.12, 0.47, 0.68, 0.91]
spin_labels = ["r1", "r2", "r3", "r4"]
selected_names = []

for i in range(len(spins)):
    spin = spins[i]
    label = spin_labels[i]

    # find which individual is selected
    for name in ["P1", "P2", "P3", "P4"]:
        lo = cum_data[name][4]
        hi = cum_data[name][5]
        if lo < spin <= hi or (spin <= hi and name == "P1"):
            selected_names.append(name)
            chrom = cum_data[name][0]
            print(f"  {label}={spin}: falls in [{lo:.4f}, {hi:.4f}) -> {name} = {chrom} selected")
            break

print(f"\n  Parent Pair 1: {selected_names[0]} and {selected_names[1]}")
print(f"  Parent Pair 2: {selected_names[2]} and {selected_names[3]}")

# STEP 4: Crossover at position 2
print()
print(" STEP 4: Single-point crossover at position 2 (after bit 2)")
print()

pair1_p1 = cum_data[selected_names[0]][0]
pair1_p2 = cum_data[selected_names[1]][0]
pair2_p1 = cum_data[selected_names[2]][0]
pair2_p2 = cum_data[selected_names[3]][0]

o1, o2 = single_point_crossover(pair1_p1, pair1_p2, 2)
o3, o4 = single_point_crossover(pair2_p1, pair2_p2, 2)

x_o1 = decode(o1)
x_o2 = decode(o2)
x_o3 = decode(o3)
x_o4 = decode(o4)

print(f"  Pair 1: {selected_names[0]}={pair1_p1} x {selected_names[1]}={pair1_p2}")
print(f"    O1 = {pair1_p1[:2]} + {pair1_p2[2:]} = {o1}  -> x={x_o1}, f(x)={fitness(o1)}")
print(f"    O2 = {pair1_p2[:2]} + {pair1_p1[2:]} = {o2}  -> x={x_o2}, f(x)={fitness(o2)}")
print()
print(f"  Pair 2: {selected_names[2]}={pair2_p1} x {selected_names[3]}={pair2_p2}")
print(f"    O3 = {pair2_p1[:2]} + {pair2_p2[2:]} = {o3}  -> x={x_o3}, f(x)={fitness(o3)}")
print(f"    O4 = {pair2_p2[:2]} + {pair2_p1[2:]} = {o4}  -> x={x_o4}, f(x)={fitness(o4)}")

# STEP 5: Mutation on O1
print()
print(" STEP 5: Bit-flip mutation on O1 using given random values, p_m=0.1")
print()

o1_pre = list(o1)
mutation_randoms = [0.08, 0.43, 0.91, 0.05]
p_m_val = 0.1

print(f"  O1 before mutation : {o1_pre}  (x={decode(o1_pre)}, f={fitness(o1_pre)})")
print()

o1_mutated = []
for bit_i in range(len(o1_pre)):
    r_val = mutation_randoms[bit_i]
    original_bit = o1_pre[bit_i]

    if r_val < p_m_val:
        new_bit = 1 - original_bit
        action = f"r={r_val} < p_m={p_m_val} -> FLIP {original_bit} to {new_bit}"
    else:
        new_bit = original_bit
        action = f"r={r_val} >= p_m={p_m_val} -> keep {original_bit}"

    o1_mutated.append(new_bit)
    print(f"  Bit {bit_i}: {action}")

print()
print(f"  O1 after mutation  : {o1_mutated}  (x={decode(o1_mutated)}, f={fitness(o1_mutated)})")


PART (d) - Manual Trace: One Complete GA Generation
           No code-based GA calls -- all arithmetic shown explicitly


Starting population and given random numbers:
  P1 = [0,1,1,0]    r1 = 0.12  (selection spin 1)
  P2 = [1,0,0,1]    r2 = 0.47  (selection spin 2)
  P3 = [1,1,0,0]    r3 = 0.68  (selection spin 3)
  P4 = [0,0,1,1]    r4 = 0.91  (selection spin 4)
  Crossover point = 2 (after bit index 2)
  Mutation on O1 : per-bit random values = [0.08, 0.43, 0.91, 0.05], p_m = 0.1

 STEP 1: Decode chromosomes and compute f(x) = -x^2 + 14x + 5

  P1 = [0, 1, 1, 0]
       decode  : 0*8 + 1*4 + 1*2 + 0*1 = 6
       f(x)    : -6^2 + 14*6 + 5 = -36 + 84 + 5 = 53
  P2 = [1, 0, 0, 1]
       decode  : 1*8 + 0*4 + 0*2 + 1*1 = 9
       f(x)    : -9^2 + 14*9 + 5 = -81 + 126 + 5 = 50
  P3 = [1, 1, 0, 0]
       decode  : 1*8 + 1*4 + 0*2 + 0*1 = 12
       f(x)    : -12^2 + 14*12 + 5 = -144 + 168 + 5 = 29
  P4 = [0, 0, 1, 1]
       decode  : 0*8 + 0*4 + 1*2 + 1*1 = 3
       f(x)    : -3^2 + 14*3 

# Q5 — Implementing a GA for the Course Scheduling Problem

In [21]:
import random

In [22]:
# Problem constants
COURSES = 6
ROOMS = 3
SLOTS = 4
TOTAL_CELLS = ROOMS * SLOTS  # 12 possible (room, slot) combinations

print("Q5: Course Scheduling Problem")
print(f"  Courses: {COURSES}, Rooms: {ROOMS}, Time slots: {SLOTS}")
print(f"  Total (room,slot) cells: {TOTAL_CELLS}")
print("  Goal: assign each course a (room, slot) with NO two courses sharing the same cell")
print("  Chromosome: list of 6 tuples [(room_idx, slot_idx), ...]")
print("  Fitness: f = 100 - (10 x conflicts).  Conflict-free = 100.")


Q5: Course Scheduling Problem
  Courses: 6, Rooms: 3, Time slots: 4
  Total (room,slot) cells: 12
  Goal: assign each course a (room, slot) with NO two courses sharing the same cell
  Chromosome: list of 6 tuples [(room_idx, slot_idx), ...]
  Fitness: f = 100 - (10 x conflicts).  Conflict-free = 100.


In [23]:
# count_conflicts: counts pairs of courses sharing the same (room, slot)
def count_conflicts(chromosome):
    conflicts = 0
    # compare every pair of courses
    for i in range(len(chromosome)):
        for j in range(i + 1, len(chromosome)):
            # if they share the same cell -> conflict
            if chromosome[i] == chromosome[j]:
                conflicts = conflicts + 1
    return conflicts

In [24]:
# fitness function for scheduling
def fitness(chromosome):
    conflicts = count_conflicts(chromosome)
    return 100 - (10 * conflicts)

In [25]:
# generate a random chromosome (may have conflicts)
def random_chromosome():
    chrom = []
    for i in range(COURSES):
        room = random.randint(0, ROOMS - 1)
        slot = random.randint(0, SLOTS - 1)
        chrom.append((room, slot))
    return chrom

In [26]:
# helper to print chromosome in a readable way
def chromosome_str(chrom):
    parts = []
    for i in range(len(chrom)):
        r = chrom[i][0]
        s = chrom[i][1]
        parts.append(f"C{i+1}(R{r+1},T{s+1})")
    return "  ".join(parts)


In [27]:
# PART (a): test the representation, count_conflicts, and fitness
print("PART (a) - Representation, count_conflicts, and fitness")
print()
print("Generating and printing 5 random chromosomes:")
print()
print(f"  {'#':<4}  {'Chromosome (C=Course, R=Room, T=TimeSlot)':<55}  {'Conflicts':>9}  {'Fitness':>7}")
print(f"  {'-'*80}")

random.seed(10)
for i in range(5):
    chrom = random_chromosome()
    conflicts = count_conflicts(chrom)
    fit = fitness(chrom)
    print(f"  {i+1:<4}  {chromosome_str(chrom):<55}  {conflicts:>9}  {fit:>7}")


PART (a) - Representation, count_conflicts, and fitness

Generating and printing 5 random chromosomes:

  #     Chromosome (C=Course, R=Room, T=TimeSlot)                Conflicts  Fitness
  --------------------------------------------------------------------------------
  1     C1(R3,T1)  C2(R2,T4)  C3(R3,T1)  C4(R1,T4)  C5(R2,T3)  C6(R3,T2)          1       90
  2     C1(R1,T4)  C2(R2,T1)  C3(R1,T3)  C4(R1,T4)  C5(R1,T3)  C6(R2,T4)          2       80
  3     C1(R2,T3)  C2(R2,T2)  C3(R3,T3)  C4(R3,T3)  C5(R1,T4)  C6(R1,T4)          2       80
  4     C1(R3,T4)  C2(R1,T1)  C3(R1,T2)  C4(R1,T3)  C5(R3,T3)  C6(R1,T3)          1       90
  5     C1(R3,T4)  C2(R2,T4)  C3(R1,T3)  C4(R3,T2)  C5(R1,T4)  C6(R1,T1)          0      100


In [29]:
# repair: fixes a conflicting chromosome
# detects courses sharing a cell, moves them to a free cell
def repair(chromosome):
    # build a map of cell -> list of courses using that cell
    cell_to_courses = {}
    for i in range(len(chromosome)):
        cell = chromosome[i]
        if cell not in cell_to_courses:
            cell_to_courses[cell] = []
        cell_to_courses[cell].append(i)

    # start with the current assignment, we'll fix conflicts
    new_chrom = list(chromosome)

    # find which cells are occupied (by one course, kept as is)
    occupied_cells = set()
    for cell in cell_to_courses:
        if len(cell_to_courses[cell]) == 1:
            occupied_cells.add(cell)

    # for each cell with multiple courses, keep the first one and fix the rest
    for cell in cell_to_courses:
        courses_in_cell = cell_to_courses[cell]
        if len(courses_in_cell) > 1:
            # keep courses_in_cell[0] in place
            occupied_cells.add(cell)

            # reassign the rest (courses_in_cell[1:])
            for course_idx in courses_in_cell[1:]:
                # find a free cell
                free_cells = []
                for r in range(ROOMS):
                    for s in range(SLOTS):
                        candidate = (r, s)
                        if candidate not in occupied_cells:
                            free_cells.append(candidate)

                if len(free_cells) > 0:
                    # pick a random free cell
                    new_cell = random.choice(free_cells)
                    new_chrom[course_idx] = new_cell
                    occupied_cells.add(new_cell)
                else:
                    # no free cell left, assign randomly
                    r = random.randint(0, ROOMS - 1)
                    s = random.randint(0, SLOTS - 1)
                    new_chrom[course_idx] = (r, s)

    return new_chrom


In [30]:
# crossover: standard single-point crossover
def crossover(p1, p2, point):
    offspring1 = []
    offspring2 = []

    for i in range(len(p1)):
        if i < point:
            offspring1.append(p1[i])
            offspring2.append(p2[i])
        else:
            offspring1.append(p2[i])
            offspring2.append(p1[i])

    return offspring1, offspring2

In [31]:
# mutate: with probability p_m per gene, reassign that course to a random cell
def mutate(chromosome, p_m):
    new_chrom = list(chromosome)
    for i in range(len(new_chrom)):
        if random.random() < p_m:
            # reassign this course to a random (room, slot)
            new_chrom[i] = (random.randint(0, ROOMS - 1), random.randint(0, SLOTS - 1))
    return new_chrom


In [32]:
# PART (b): demonstrate crossover creating a conflict, then repair fixing it
print("PART (b) - Crossover, repair, mutate (with demonstrations)")
print()

print("""
Three operators implemented:
  crossover(p1, p2, point) : single-point crossover; offspring may conflict.
  repair(chromosome)       : detects conflicting courses, reassigns to free cells.
  mutate(chromosome, p_m)  : per-gene random reassignment at rate p_m.
""")

# DEMONSTRATION 1: how crossover creates a conflict
print(" Demonstration 1: How crossover creates a conflict")
print()

# two conflict-free parent chromosomes
parent_a = [(0,0),(1,1),(2,2),(0,3),(1,2),(2,1)]
parent_b = [(2,0),(0,1),(1,2),(2,3),(0,2),(1,1)]

ca = count_conflicts(parent_a)
cb = count_conflicts(parent_b)

print(f"  Parent A : {chromosome_str(parent_a)}")
print(f"             conflicts = {ca},  f = {fitness(parent_a)}")
print(f"  Parent B : {chromosome_str(parent_b)}")
print(f"             conflicts = {cb},  f = {fitness(parent_b)}")

# crossover at point 3
point = 3
o1, o2 = crossover(parent_a, parent_b, point)
co1 = count_conflicts(o1)
co2 = count_conflicts(o2)

print(f"\n  Crossover at point = {point}:")
print(f"    Offspring1 = A[:3] + B[3:] = {parent_a[:point]} + {parent_b[point:]}")
print(f"    Offspring1 : {chromosome_str(o1)}")
print(f"               conflicts = {co1},  f = {fitness(o1)}")

# show which courses conflict
cell_map = {}
for i in range(len(o1)):
    cell = o1[i]
    if cell not in cell_map:
        cell_map[cell] = []
    cell_map[cell].append(f"C{i+1}")

for cell in cell_map:
    if len(cell_map[cell]) > 1:
        r = cell[0]
        s = cell[1]
        print(f"    -> Conflict: {' and '.join(cell_map[cell])} both assigned to (R{r+1},T{s+1})")

print(f"\n    Offspring2 = B[:3] + A[3:] = {parent_b[:point]} + {parent_a[point:]}")
print(f"    Offspring2 : {chromosome_str(o2)}")
print(f"               conflicts = {co2},  f = {fitness(o2)}")

cell_map2 = {}
for i in range(len(o2)):
    cell = o2[i]
    if cell not in cell_map2:
        cell_map2[cell] = []
    cell_map2[cell].append(f"C{i+1}")

for cell in cell_map2:
    if len(cell_map2[cell]) > 1:
        r = cell[0]
        s = cell[1]
        print(f"    -> Conflict: {' and '.join(cell_map2[cell])} both assigned to (R{r+1},T{s+1})")


PART (b) - Crossover, repair, mutate (with demonstrations)


Three operators implemented:
  crossover(p1, p2, point) : single-point crossover; offspring may conflict.
  repair(chromosome)       : detects conflicting courses, reassigns to free cells.
  mutate(chromosome, p_m)  : per-gene random reassignment at rate p_m.

 Demonstration 1: How crossover creates a conflict

  Parent A : C1(R1,T1)  C2(R2,T2)  C3(R3,T3)  C4(R1,T4)  C5(R2,T3)  C6(R3,T2)
             conflicts = 0,  f = 100
  Parent B : C1(R3,T1)  C2(R1,T2)  C3(R2,T3)  C4(R3,T4)  C5(R1,T3)  C6(R2,T2)
             conflicts = 0,  f = 100

  Crossover at point = 3:
    Offspring1 = A[:3] + B[3:] = [(0, 0), (1, 1), (2, 2)] + [(2, 3), (0, 2), (1, 1)]
    Offspring1 : C1(R1,T1)  C2(R2,T2)  C3(R3,T3)  C4(R3,T4)  C5(R1,T3)  C6(R2,T2)
               conflicts = 1,  f = 90
    -> Conflict: C2 and C6 both assigned to (R2,T2)

    Offspring2 = B[:3] + A[3:] = [(2, 0), (0, 1), (1, 2)] + [(0, 3), (1, 2), (2, 1)]
    Offspring2 : C1(R3,T1)

In [33]:
# DEMONSTRATION 2: repair() fixes a conflicting chromosome
print(" Demonstration 2: repair() fixes a conflicting chromosome")
print()

random.seed(7)

# manually create a chromosome with known conflicts
# C1 and C3 both in (R1,T1); C4 and C6 both in (R2,T2)
conflicted = [(0,0),(1,2),(0,0),(1,1),(2,3),(1,1)]
c_before = count_conflicts(conflicted)
f_before = fitness(conflicted)

print(f"  Conflicted chromosome (manually constructed):")
print(f"  {chromosome_str(conflicted)}")
print(f"  conflicts before repair = {c_before},  f = {f_before}")

# show which pairs are conflicting
for i in range(COURSES):
    for j in range(i + 1, COURSES):
        if conflicted[i] == conflicted[j]:
            r = conflicted[i][0]
            s = conflicted[i][1]
            print(f"  -> Conflict: C{i+1} and C{j+1} share (R{r+1},T{s+1})")

repaired = repair(conflicted)
c_after = count_conflicts(repaired)
f_after = fitness(repaired)

print(f"\n  After repair():")
print(f"  {chromosome_str(repaired)}")
print(f"  conflicts after repair  = {c_after},  f = {f_after}")
if c_after == 0:
    print(f"  Result: CONFLICT-FREE schedule achieved by repair!")
else:
    print(f"  Result: conflicts reduced from {c_before} to {c_after}.")

print(f"""
  How repair() works:
    1. Build an occupancy map: each (room,slot) cell -> list of course indices.
    2. In any cell with multiple courses, keep the first (index 0) and flag
       the rest as "conflicting" -- they must be moved.
    3. For each conflicting course, collect all (room,slot) cells not currently
       occupied by any settled course.  Assign a random free cell.
    4. If no free cell exists (all 12 cells taken), fall back to a random cell.
    repair() is applied after EVERY crossover in the scheduling GA to ensure
    offspring are at least partially feasible before selection.
""")

# DEMONSTRATION 3: mutate()
print(" Demonstration 3: mutate(chromosome, p_m)")
print()

base = [(0,0),(1,1),(2,2),(0,3),(1,2),(2,1)]
print(f"  Base chromosome : {chromosome_str(base)}")
print(f"  conflicts = {count_conflicts(base)},  f = {fitness(base)}")
print()
print(f"  {'p_m':<6}  {'Mutated chromosome':<60}  {'Conflicts':>9}  {'f':>5}")

for p_m in [0.0, 0.1, 0.5, 1.0]:
    random.seed(20)
    mut = mutate(base, p_m)
    c_mut = count_conflicts(mut)
    f_mut = fitness(mut)
    changed = 0
    for idx in range(len(base)):
        if base[idx] != mut[idx]:
            changed = changed + 1
    print(f"  {p_m:<6}  {chromosome_str(mut):<60}  {c_mut:>9}  {f_mut:>5}  ({changed} gene(s) changed)")


 Demonstration 2: repair() fixes a conflicting chromosome

  Conflicted chromosome (manually constructed):
  C1(R1,T1)  C2(R2,T3)  C3(R1,T1)  C4(R2,T2)  C5(R3,T4)  C6(R2,T2)
  conflicts before repair = 2,  f = 80
  -> Conflict: C1 and C3 share (R1,T1)
  -> Conflict: C4 and C6 share (R2,T2)

  After repair():
  C1(R1,T1)  C2(R2,T3)  C3(R2,T4)  C4(R2,T2)  C5(R3,T4)  C6(R1,T3)
  conflicts after repair  = 0,  f = 100
  Result: CONFLICT-FREE schedule achieved by repair!

  How repair() works:
    1. Build an occupancy map: each (room,slot) cell -> list of course indices.
    2. In any cell with multiple courses, keep the first (index 0) and flag
       the rest as "conflicting" -- they must be moved.
    3. For each conflicting course, collect all (room,slot) cells not currently
       occupied by any settled course.  Assign a random free cell.
    4. If no free cell exists (all 12 cells taken), fall back to a random cell.
    repair() is applied after EVERY crossover in the scheduling GA t

In [34]:
print(f"""
  Why crossover introduces conflicts:
    Parent A's left genes and Parent B's right genes are designed to be
    individually conflict-free but they were optimised independently.
    When the prefix of A is joined with the suffix of B, the same
    (room,slot) cell can appear twice -- once inherited from A and once
    from B -- because neither parent "knew" what the other would contribute.
    This is the fundamental tension in constraint-satisfaction GAs: operators
    that work on independent gene blocks cannot guarantee global feasibility.
""")



  Why crossover introduces conflicts:
    Parent A's left genes and Parent B's right genes are designed to be
    individually conflict-free but they were optimised independently.
    When the prefix of A is joined with the suffix of B, the same
    (room,slot) cell can appear twice -- once inherited from A and once
    from B -- because neither parent "knew" what the other would contribute.
    This is the fundamental tension in constraint-satisfaction GAs: operators
    that work on independent gene blocks cannot guarantee global feasibility.



In [35]:
# PART (c): Full Scheduling GA

# tournament selection: pick k random individuals, return the fittest
def tournament_select(population, k=2):
    contestants = random.sample(population, k)

    # find the fittest among the contestants
    best = contestants[0]
    best_fit = fitness(best)

    for i in range(1, len(contestants)):
        f_val = fitness(contestants[i])
        if f_val > best_fit:
            best_fit = f_val
            best = contestants[i]

    return best


In [36]:
# Full scheduling GA
def run_scheduling_ga(pop_size, generations, p_m, verbose=True):
    """
    Full GA for Course Scheduling.
    - Tournament selection (size=2)
    - Single-point crossover at random gene index
    - repair() applied after every crossover
    - mutate() applied to all offspring
    Returns (best_chromosome, best_fitness, fitness_history, solution_gen)
    """

    # initialise population - repair all starting chromosomes
    population = []
    for i in range(pop_size):
        chrom = repair(random_chromosome())
        population.append(chrom)

    fitness_history = []
    solution_gen = None
    best_ever = None
    best_ever_f = -1

    for gen in range(1, generations + 1):

        # evaluate all individuals
        # find best in this generation
        gen_best_chrom = population[0]
        gen_best_f = fitness(population[0])

        for chrom in population:
            f_val = fitness(chrom)
            if f_val > gen_best_f:
                gen_best_f = f_val
                gen_best_chrom = chrom

        fitness_history.append(gen_best_f)

        # update best ever found
        if gen_best_f > best_ever_f:
            best_ever_f = gen_best_f
            best_ever = list(gen_best_chrom)

        # check if we found a solution
        if gen_best_f == 100 and solution_gen is None:
            solution_gen = gen
            if verbose:
                print(f"  Solution found at generation {gen}: {gen_best_chrom}")

        # print progress
        if verbose:
            if gen <= 5 or gen % 10 == 0 or gen == generations:
                total_f = 0
                for chrom in population:
                    total_f = total_f + fitness(chrom)
                avg_f = total_f / pop_size
                print(f"  Gen {gen:>3} | best f = {gen_best_f:>4} | avg f = {avg_f:>6.2f} | best chrom = {gen_best_chrom}")

        # build next generation
        next_pop = []
        while len(next_pop) < pop_size:
            # select two parents via tournament
            p1 = tournament_select(population, k=2)
            p2 = tournament_select(population, k=2)

            # crossover at random gene position
            point = random.randint(1, COURSES - 1)
            o1, o2 = crossover(p1, p2, point)

            # repair and then mutate
            o1 = mutate(repair(o1), p_m)
            o2 = mutate(repair(o2), p_m)

            next_pop.append(o1)
            if len(next_pop) < pop_size:
                next_pop.append(o2)

        population = next_pop

    return best_ever, best_ever_f, fitness_history, solution_gen


In [37]:
# PART (c): Run the full scheduling GA
print()
print("=" * 75)
print("PART (c) - Full Scheduling GA")
print("           pop_size=20, generations=50, p_m=0.1")
print("=" * 75)
print()

print("""
Algorithm:
  Selection  : Tournament (size=2) -- pick 2 random individuals, keep fitter one.
  Crossover  : Single-point at random gene index in [1, 5].
  Repair     : Applied to EVERY offspring immediately after crossover.
  Mutation   : Per-gene random reassignment at rate p_m=0.1.
  Population : Initialised with repair() so all starting chromosomes are
               as feasible as possible before evolution begins.
""")

random.seed(42)
print("--- GA Run: pop_size=20, generations=50, p_m=0.1 (seed=42) ---")
print()

best_chrom, best_f, history, sol_gen = run_scheduling_ga(pop_size=20, generations=50, p_m=0.1, verbose=True)

# print best schedule
print()
print(" Best Schedule Found")
print()
print(f"  Fitness          : {best_f}")
print(f"  Conflicts        : {count_conflicts(best_chrom)}")
print(f"  Chromosome       : {best_chrom}")

if sol_gen is not None:
    print(f"  Solution at gen  : {sol_gen}")
else:
    print(f"  Solution found   : No (0-conflict schedule not achieved in this run)")

print()
print("  Course assignments:")
print(f"  {'Course':<10}  {'Room':>5}  {'Time Slot':>10}")
for i in range(len(best_chrom)):
    r = best_chrom[i][0]
    s = best_chrom[i][1]
    print(f"  C{i+1:<9}  R{r+1:>5}  T{s+1:>10}")

# check for remaining conflicts
cell_map = {}
for i in range(len(best_chrom)):
    cell = best_chrom[i]
    if cell not in cell_map:
        cell_map[cell] = []
    cell_map[cell].append(f"C{i+1}")

conflict_pairs = []
for cell in cell_map:
    if len(cell_map[cell]) > 1:
        conflict_pairs.append((cell, cell_map[cell]))

if len(conflict_pairs) > 0:
    print()
    print("  Remaining conflicts:")
    for pair in conflict_pairs:
        r = pair[0][0]
        s = pair[0][1]
        print(f"    {' and '.join(pair[1])} share (R{r+1},T{s+1})")
else:
    print()
    print("  No conflicts -- schedule is fully feasible!")

# fitness progression
print()
print(" Fitness Progression Per Generation")
print()
print(f"  {'Gen':>4}  {'Best f':>7}  {'Bar'}")
for gen in range(len(history)):
    f_val = history[gen]
    if (gen + 1) <= 10 or (gen + 1) % 5 == 0:
        bar = "#" * (f_val // 5)
        print(f"  {gen+1:>4}  {f_val:>7}  {bar}")



PART (c) - Full Scheduling GA
           pop_size=20, generations=50, p_m=0.1


Algorithm:
  Selection  : Tournament (size=2) -- pick 2 random individuals, keep fitter one.
  Crossover  : Single-point at random gene index in [1, 5].
  Repair     : Applied to EVERY offspring immediately after crossover.
  Mutation   : Per-gene random reassignment at rate p_m=0.1.
  Population : Initialised with repair() so all starting chromosomes are
               as feasible as possible before evolution begins.

--- GA Run: pop_size=20, generations=50, p_m=0.1 (seed=42) ---

  Solution found at generation 1: [(2, 0), (0, 2), (0, 1), (0, 0), (0, 3), (2, 3)]
  Gen   1 | best f =  100 | avg f =  99.50 | best chrom = [(2, 0), (0, 2), (0, 1), (0, 0), (0, 3), (2, 3)]
  Gen   2 | best f =  100 | avg f =  98.00 | best chrom = [(1, 2), (1, 3), (0, 2), (0, 0), (0, 3), (2, 3)]
  Gen   3 | best f =  100 | avg f =  97.00 | best chrom = [(2, 3), (0, 0), (0, 3), (2, 1), (1, 1), (2, 0)]
  Gen   4 | best f =  100 | 

In [38]:
# Reliability test: 10 independent runs
print()
print(" Reliability Test: 10 Independent Runs")
print()
print(f"  {'Run':>4}  {'Best f':>7}  {'Conflicts':>10}  {'Sol Gen':>8}  {'Found?':>7}")

successes = 0
sol_gens = []

for run in range(1, 11):
    random.seed(run * 17)
    bc, bf, _, sg = run_scheduling_ga(20, 50, 0.1, verbose=False)
    cc = count_conflicts(bc)

    if bf == 100:
        found = "YES"
        successes = successes + 1
        if sg is not None:
            sol_gens.append(sg)
        sg_str = str(sg) if sg is not None else "N/A"
    else:
        found = "No"
        sg_str = "N/A"

    print(f"  {run:>4}  {bf:>7}  {cc:>10}  {sg_str:>8}  {found:>7}")

print(f"\n  Conflict-free solutions found : {successes}/10")

if len(sol_gens) > 0:
    total_sg = 0
    for sg in sol_gens:
        total_sg = total_sg + sg
    avg_sg = total_sg / len(sol_gens)
    print(f"  Avg generation to solution   : {avg_sg:.1f}")
else:
    print(f"  Avg generation to solution   : N/A (no solutions found)")



 Reliability Test: 10 Independent Runs

   Run   Best f   Conflicts   Sol Gen   Found?
     1      100           0         1      YES
     2      100           0         1      YES
     3      100           0         1      YES
     4      100           0         1      YES
     5      100           0         1      YES
     6      100           0         1      YES
     7      100           0         1      YES
     8      100           0         1      YES
     9      100           0         1      YES
    10      100           0         1      YES

  Conflict-free solutions found : 10/10
  Avg generation to solution   : 1.0


In [39]:
# PART (d): Written Analysis
import math

total_chromosomes = TOTAL_CELLS ** COURSES

# number of valid conflict-free schedules = P(12, 6)
valid_schedules = 1
for i in range(COURSES):
    valid_schedules = valid_schedules * (TOTAL_CELLS - i)

valid_fraction = valid_schedules / total_chromosomes

print("PART (d) - Written Analysis")
print(f"""
--- Part (d.1): Did the GA consistently find a conflict-free schedule? ---

  Total possible chromosomes  : {ROOMS}^COURSES * {SLOTS}^COURSES
                              = 12^{COURSES} = {total_chromosomes:,}
  Conflict-free schedules     : P(12,{COURSES}) = 12*11*10*9*8*7 = {valid_schedules:,}
  Fraction of valid schedules : {valid_schedules:,} / {total_chromosomes:,}
                              = {valid_fraction:.4f}  ({valid_fraction*100:.1f}%)

  From the 10 independent runs above, the GA found conflict-free schedules
  in the majority of trials. When it does NOT find a solution, the cause is
  premature convergence: once the population clusters around a near-optimal
  but conflicted chromosome, mutation at p_m=0.1 may not generate the exact
  gene change needed to move the last conflicting course to a free cell.
  The constraint that makes this hardest is the room-slot UNIQUENESS rule:
  with 6 courses competing for 12 distinct (room,slot) cells, the final
  assignment requires a perfect matching. As the GA converges, the repair
  operator must work within an increasingly constrained residual space,
  and with only 20 individuals in the population, genetic diversity
  sometimes collapses before the last collision is resolved.

  The valid-schedule fraction is {valid_fraction*100:.1f}% -- much higher than many
  combinatorial problems -- which is why the GA succeeds most of the time.
  If the problem had fewer cells (e.g. 3 slots instead of 4), the valid
  fraction would drop to P(9,6)/9^6 = 60,480/531,441 = 11.4%, making it
  significantly harder and increasing the failure rate.

--- Part (d.2): GA vs RRHC -- when is each preferable? ---

  TWO CONDITIONS WHERE THIS GA IS PREFERABLE TO RRHC:
  1. Large search space with combinatorial structure:
     The scheduling problem has 12^6 = 2,985,984 possible chromosomes.
     RRHC evaluates one random starting point per restart and climbs
     greedily -- it is blind to the population-level patterns that the
     GA exploits. The GA's crossover operator recombines PARTIAL solutions
     (e.g., a conflict-free assignment for C1-C3 from one parent, and
     a conflict-free assignment for C4-C6 from another), assembling
     feasible sub-schedules into full solutions. RRHC cannot recombine
     information from different search trajectories.

  2. Dense constraint interaction between genes:
     In this problem, assigning one course's room and slot affects the
     feasibility of all other courses (any shared cell is a conflict).
     The GA's population maintains DIVERSE candidate solutions
     simultaneously, allowing crossover to find synergistic combinations.
     RRHC with a swap neighbourhood evaluates one candidate at a time;
     its local neighbourhood is blind to improvements that require
     simultaneously moving two or more courses.

  TWO CONDITIONS WHERE RRHC IS PREFERABLE TO THIS GA:
  1. Very small search space or when any feasible solution is acceptable:
     If the number of courses were 3 instead of 6, the valid-schedule
     fraction rises to P(12,3)/12^3 = 1320/1728 = 76.4%. In that case
     a random restart approach finds a conflict-free schedule almost
     immediately (within 1-2 restarts on average), while the GA wastes
     effort maintaining a population of 20 chromosomes, running selection,
     crossover, and mutation for 50 generations unnecessarily.

  2. When implementation simplicity and runtime speed matter most:
     RRHC for this problem requires only a random_chromosome() function
     and a simple greedy assignment step (assign courses to unused cells).
     The GA requires five components (selection, crossover, repair,
     mutation, population management) and runs pop_size * generations =
     20 * 50 = 1,000 evaluations. For problems where the constraint
     density is low (few conflicts in random chromosomes) or where a
     good-enough (not optimal) solution suffices, RRHC's simplicity and
     low overhead make it the practical choice.
""")


PART (d) - Written Analysis

--- Part (d.1): Did the GA consistently find a conflict-free schedule? ---

  Total possible chromosomes  : 3^COURSES * 4^COURSES
                              = 12^6 = 2,985,984
  Conflict-free schedules     : P(12,6) = 12*11*10*9*8*7 = 665,280
  Fraction of valid schedules : 665,280 / 2,985,984
                              = 0.2228  (22.3%)

  From the 10 independent runs above, the GA found conflict-free schedules
  in the majority of trials. When it does NOT find a solution, the cause is
  premature convergence: once the population clusters around a near-optimal
  but conflicted chromosome, mutation at p_m=0.1 may not generate the exact
  gene change needed to move the last conflicting course to a free cell.
  The constraint that makes this hardest is the room-slot UNIQUENESS rule:
  with 6 courses competing for 12 distinct (room,slot) cells, the final
  assignment requires a perfect matching. As the GA converges, the repair
  operator must work within